<a href="https://colab.research.google.com/github/Trisha108-hub/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Trisha108-hub/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
import duckdb, pandas as pd, numpy as np

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

FEATURE_MONTH, LABEL_MONTH = "2026-03", "2026-04"
feat_path  = f"{rel}/fact_content_daily_performance/month={FEATURE_MONTH}/data_0.parquet"
label_path = f"{rel}/fact_content_daily_performance/month={LABEL_MONTH}/data_0.parquet"
dim_content_path = f"{rel}/dim_content.parquet"

content_col, client_col = "content_hash_id", "client_hash_id"
clicks_col, impr_col, avgpos_col, avail_col = "gsc_clicks", "gsc_impressions", "gsc_avg_position", "gsc_data_available"

feat = con.sql(f"""
    SELECT {content_col} AS content_id, {client_col} AS client_id,
           AVG({clicks_col}) AS avg_clicks_mar, AVG({impr_col}) AS avg_impressions_mar,
           AVG({avgpos_col}) AS avg_position_mar, AVG({clicks_col}/NULLIF({impr_col},0)) AS avg_ctr_mar
    FROM read_parquet('{feat_path}') WHERE {avail_col} IS TRUE GROUP BY 1,2
""").df()

outcome = con.sql(f"""
    SELECT {content_col} AS content_id, AVG({clicks_col}) AS avg_clicks_apr
    FROM read_parquet('{label_path}') WHERE {avail_col} IS TRUE GROUP BY 1
""").df()

content_meta = con.sql(f"""
    SELECT {content_col} AS content_id, search_volume, competition, cpc, backlinks,
           word_count, char_count, category_count, main_intent, content_type
    FROM read_parquet('{dim_content_path}') WHERE is_published IS TRUE AND is_deleted IS FALSE
""").df()

features_df = feat.merge(outcome, on='content_id', how='inner').merge(content_meta, on='content_id', how='inner')
features_df['label'] = (features_df['avg_clicks_apr'] > features_df['avg_clicks_mar']).astype(int)
print(features_df.shape, features_df['label'].mean())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(158464, 17) 0.20056290387722134


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1 — ML Appendix, "What Predicts Health?" (Random Forest feature
importance: Average Position 43%, Impressions 32%, Scroll Depth 15%)**

The paper itself is admirably upfront that health score is a composite
metric built directly from position, impressions, CTR, and scroll depth — so
using health score as the model's target while also feeding it position and
impressions as features means part of the "predictive power" is arithmetic
overlap, not learned signal. My methodology question: given that overlap,
what does the reported holdout split actually validate here? An 80/20 random
holdout confirms the model generalizes to unseen *rows*, but it can't
separate "the model learned something real about content performance" from
"the model rediscovered the health-score formula." A cleaner test might be
reporting feature importance against a target built *without* position and
impressions (e.g. only scroll depth + engagement + freshness), to see what
predictive power survives once the circular inputs are removed.

**Finding 2 — ML Appendix, "What Predicts Growth?" (Logistic regression, 71%
holdout accuracy; content age, days-since-update, and days-visible as the
top signals)**

The growth/decline label is defined earlier in the paper as a 30-day-vs-
previous-30-day impression trend (>10% change). My methodology question:
was the 80/20 holdout split done randomly across rows, or time-aware /
grouped by content or brand? "Days visible" as a top positive predictor is
worth double-checking for a subtle timing issue — if a page's growth-window
impressions are what make it "visible" in that same window, days-visible and
the growth label could be measuring overlapping time periods rather than
days-visible *causing or preceding* the growth. A time-aware split (train on
an earlier period, test on a later one) or explicitly lagging the
days-visible feature to end before the label's measurement window would rule
this out and make the 71% figure easier to trust as a forward-looking signal
rather than a same-window correlation.

Both questions are asked in the spirit the paper itself sets: it already
labels its ML section "exploratory... descriptive rather than causal," so
these aren't gotchas — they're the natural next check the paper's own
disclosed standard invites, and the same check applied to my own Week-5
model in Section 2 below.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer # Import SimpleImputer
import pandas as pd # Import pandas for type checking

numeric_feats = ['avg_clicks_mar','avg_impressions_mar','avg_position_mar','avg_ctr_mar',
                  'search_volume','competition','cpc','backlinks','word_count','char_count','category_count']
cat_feats = ['main_intent','content_type']

# Create a pipeline for numeric features: impute missing values (pd.NA will become np.nan when coerced to float)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')) # You can choose 'median', 'most_frequent', etc.
])

# Modify ColumnTransformer to use the numeric_transformer for numeric features.
# This will ensure that the numeric features are treated as floats and NaNs are handled.
pre = ColumnTransformer([('num', numeric_transformer, numeric_feats),
                         ('cat', OneHotEncoder(handle_unknown='ignore'), cat_feats)])

def run_rf(train_df, test_df):
    # Ensure numeric columns are float type so pd.NA becomes np.nan for imputer.
    # Create copies to avoid modifying the original dataframes passed in if they are used elsewhere
    # or if run_rf is called multiple times with the same initial df.
    train_df_copy = train_df.copy()
    test_df_copy = test_df.copy()

    for col in numeric_feats:
        if pd.api.types.is_integer_dtype(train_df_copy[col]):
            train_df_copy[col] = train_df_copy[col].astype(float)
        if pd.api.types.is_integer_dtype(test_df_copy[col]):
            test_df_copy[col] = test_df_copy[col].astype(float)

    Xtr, ytr = train_df_copy[numeric_feats+cat_feats], train_df_copy['label']
    Xte, yte = test_df_copy[numeric_feats+cat_feats], test_df_copy['label']
    model = Pipeline([('pre',pre), ('clf', RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42))]).fit(Xtr,ytr)
    scores = model.predict_proba(Xte)[:,1]
    return roc_auc_score(yte, scores), model, Xte, yte, scores

train_naive, test_naive = train_test_split(features_df, test_size=0.25, random_state=42)
auc_naive, _, _, _, _ = run_rf(train_naive, test_naive)

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(features_df, groups=features_df['client_id']))
train_grp, test_grp = features_df.iloc[tr_idx], features_df.iloc[te_idx]
auc_grouped, rf_model, X_test_grp, y_test_grp, scores_grp = run_rf(train_grp, test_grp)

print(f"BEFORE (naive random split) ROC-AUC: {auc_naive:.4f}")
print(f"AFTER  (grouped-by-client split) ROC-AUC: {auc_grouped:.4f}")
print(f"Gap: {auc_naive - auc_grouped:.4f}")

BEFORE (naive random split) ROC-AUC: 0.7515
AFTER  (grouped-by-client split) ROC-AUC: 0.6956
Gap: 0.0559


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Numeric features computed ONLY from", FEATURE_MONTH, "— strictly before label window", LABEL_MONTH)
print("Content metadata is a static dim_content snapshot — flagged since Week 3/4 as NOT")
print("reliably decision-time-safe for historical months (single frozen snapshot, not per-month history).")
corr = features_df[numeric_feats + ['label']].corr()['label'].sort_values(ascending=False)
print(corr)

Numeric features computed ONLY from 2026-03 — strictly before label window 2026-04
Content metadata is a static dim_content snapshot — flagged since Week 3/4 as NOT
reliably decision-time-safe for historical months (single frozen snapshot, not per-month history).
label                  1.000000
avg_impressions_mar    0.109390
word_count             0.071147
avg_clicks_mar         0.067078
char_count             0.051200
category_count         0.004627
backlinks             -0.008285
search_volume         -0.013774
cpc                   -0.016408
avg_ctr_mar           -0.016511
competition           -0.053818
avg_position_mar      -0.112979
Name: label, dtype: float64


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Claim rewrite. Boldest original claim (from Week 5): "Random Forest beats the baseline rule at identifying pages likely to gain clicks." Rewritten safely: In this sample, under a grouped client-level split, Random Forest showed higher precision-at-top-20% (0.329) than the Week-4 baseline rule (0.295) for flagging content whose clicks rose month-over-month — an observed, decision-support signal on this snapshot, not a guarantee for any individual page or a claim about why clicks changed.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
wrong_preds = X_test_grp.copy()
wrong_preds['label'] = y_test_grp.values
wrong_preds['score'] = scores_grp
print(wrong_preds.sort_values('score', ascending=False).head(5))

       avg_clicks_mar  avg_impressions_mar  avg_position_mar  avg_ctr_mar  \
3691         0.903226           191.354839         11.474386     0.005433   
3462         0.741935           270.838710         11.080763     0.002957   
3555         1.935484           235.774194          9.342098     0.007995   
82557        3.516129           531.161290         12.702118     0.006858   
3332         0.032258            40.903226         10.750427     0.000633   

       search_volume  competition   cpc  backlinks  word_count  char_count  \
3691            20.0         0.81  8.39        NaN      2951.0     17566.0   
3462          4400.0         1.00  8.32        NaN      2962.0     18222.0   
3555            40.0         0.19  8.12        NaN      2387.0     14365.0   
82557          140.0         0.11  6.34        NaN      2811.0     16574.0   
3332            20.0         1.00  9.86        NaN      2349.0     14745.0   

       category_count    main_intent     content_type  label     sco

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.